# Step 4: Preprocessing and feature engineering

The code lives in `src/aq.py` so that Steps 5-7 use exactly the same pipeline. This notebook
runs each stage, shows the evidence behind each choice, and checks for leakage.

Order (it prevents leakage): **time split -> transform with train statistics -> imputation fitted on
train -> features built per city -> targets.**

Cities: the 17 modelling cities from Step 3 (at least 200 AQI days in 2015-2018). Ahmedabad is kept
(team decision); each later step also reports results without it.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:                                   # Colab: get the repo (code + data) first
    import google.colab  # noqa: F401
    if not (ROOT / "src" / "aq.py").exists():
        import subprocess
        subprocess.run(["git", "clone", "-b", "claude/wizardly-meitner-nm2sfq",
                        "https://github.com/s-aura-bhhh/Air-quality-analysis.git"], check=True)
        ROOT = Path.cwd() / "Air-quality-analysis"
except ImportError:
    pass
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import aq

FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"04_{name}.png", dpi=150); plt.show()

## 4.1 Time split

In [2]:
df = aq.load_checked()
cities = aq.model_cities(df)
d = df[df["City"].isin(cities)].reset_index(drop=True)
d["period"] = aq.period(d["Date"])
split = d.groupby("period").agg(rows=("AQI", "size"), aqi_days=("AQI", "count"),
                                first=("Date", "min"), last=("Date", "max"),
                                cities=("City", "nunique")).loc[["train", "val", "test"]]
split.to_csv(TAB / "04_split.csv")
split

,rows,aqi_days,first,last,cities
period,,,,,
train,15978,13131,2015-01-01,2018-12-31,17
val,6205,5921,2019-01-01,2019-12-31,17
test,3111,2980,2020-01-01,2020-07-01,17


Train 2015-2018, validation 2019, test 2020-01-01 to 2020-07-01. Rows are never shuffled across
time. The test period contains the lockdown (March-May 2020), so test results are also reported
with those months removed.

## 4.2 Transform: log1p where it helps, then standardise (train statistics)

Rule: apply log1p to a pollutant if it lowers |skewness| on the training rows. Then subtract the
train mean and divide by the train standard deviation. Imputation happens on this scale because
KNN and SVD rely on distances and variances, which are meaningless when columns have different
units (CO is in mg/m3) and long right tails.

In [3]:
tr = d["period"] == "train"
ls = aq.LogStandardiser().fit(d.loc[tr, aq.POLLUTANTS])
sk = pd.DataFrame({"skew_raw": d.loc[tr, aq.POLLUTANTS].skew(),
                   "skew_log1p": np.log1p(d.loc[tr, aq.POLLUTANTS]).skew()}).round(2)
sk["log_applied"] = sk.index.isin(ls.log_cols)
sk.to_csv(TAB / "04_log_decision.csv")
Z = ls.transform(d[aq.POLLUTANTS])
print("train means after standardising (should be 0):", Z[tr].mean().abs().max().round(6))
sk

train means after standardising (should be 0): 0.0


,skew_raw,skew_log1p,log_applied
PM2.5,2.94,0.08,True
PM10,1.71,-0.92,True
NO,3.48,0.51,True
NO2,2.25,-0.33,True
NOx,2.37,-0.85,True
NH3,3.79,-0.11,True
CO,8.17,2.13,True
SO2,3.87,0.78,True
O3,1.48,-0.69,True
Benzene,22.79,0.98,True


## 4.3 Imputation: which method fills missing values best?

Experiment: take the validation rows (2019), hide 10% of the values that *are* observed, fill them
with each imputer (fitted on train only), and measure RMSE between filled and true values in
standardised units. An RMSE of 1 is no better than filling with the overall mean.

Methods:
* `city_median`: each city's train median (baseline).
* `knn_k`: KNN imputation on the pollutants, k neighbours.
* `svd_k`: low-rank SVD imputation, rank k. Missing cells start at 0 (the column mean), then the
  matrix is repeatedly replaced by its rank-k reconstruction on the missing cells only.
* `city_knn_k`, `city_svd_k`: subtract each city's median first, impute the remainder with KNN or
  SVD, add the median back. This combines the city level with the same-day correlation between
  pollutants.

Repeated with two random maskings to check the ranking is stable.

In [4]:
cand = {"city_median": aq.CityMedianImputer,
        **{f"knn_{k}": (lambda k=k: aq.KNNImputerDF(k)) for k in [10, 40]},
        **{f"svd_{k}": (lambda k=k: aq.SVDImputer(k)) for k in [1, 2, 3]},
        **{f"city_knn_{k}": (lambda k=k: aq.CityCentred(aq.KNNImputerDF(k))) for k in [20, 40, 80]},
        **{f"city_svd_{k}": (lambda k=k: aq.CityCentred(aq.SVDImputer(k))) for k in [1, 2, 3]}}
res = [aq.masking_experiment(Z, d["City"], tr, cand, seed=s) for s in (42, 7)]
imp_tab = res[0].copy()
imp_tab.insert(1, "overall_seed2", res[1]["overall"])
imp_tab = imp_tab.sort_values("overall").round(3)
imp_tab.to_csv(TAB / "04_imputation_rmse.csv")
imp_tab

,overall,overall_seed2,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene
city_knn_80,0.729,0.748,0.694,0.687,0.688,0.625,0.665,0.834,0.650,0.857,0.821,0.753,0.694
city_knn_40,0.751,0.768,0.708,0.711,0.688,0.646,0.731,0.840,0.678,0.897,0.848,0.755,0.700
city_svd_1,0.754,0.780,0.733,0.736,0.739,0.685,0.624,0.823,0.548,0.803,0.854,0.873,0.836
city_svd_2,0.780,0.785,0.740,0.850,0.687,0.696,0.704,0.811,0.617,0.793,0.847,1.001,0.800
city_knn_20,0.786,0.805,0.728,0.750,0.705,0.692,0.821,0.853,0.674,0.945,0.916,0.768,0.713
knn_40,0.792,0.807,0.829,0.641,0.674,0.694,0.757,0.880,0.757,0.909,0.906,0.867,0.703
svd_1,0.811,0.824,0.777,0.744,0.799,0.743,0.576,0.860,0.764,0.895,0.936,0.945,0.822
city_median,0.818,0.835,0.871,0.872,0.859,0.775,0.711,0.867,0.568,0.876,0.837,0.860,0.861
svd_2,0.862,0.850,0.745,0.809,0.805,0.807,0.595,0.975,0.920,0.867,0.951,1.163,0.736
knn_10,0.874,0.889,0.909,0.713,0.699,0.768,0.792,0.931,0.880,1.018,1.090,0.921,0.750


In [5]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(imp_tab))
ax.bar(x - 0.2, imp_tab["overall"], 0.4, label="masking seed 1")
ax.bar(x + 0.2, imp_tab["overall_seed2"], 0.4, label="masking seed 2")
ax.set_xticks(x); ax.set_xticklabels(imp_tab.index, rotation=45, ha="right")
ax.axhline(1, color="grey", ls="--", lw=1); ax.set_ylabel("RMSE on hidden values (std units)")
ax.set_title("Imputation comparison on validation rows (lower is better)"); ax.legend()
save("imputation_rmse")

In [6]:
# Why does a low rank do best? Singular values of the train matrix (city-centred, mean-filled).
cm = aq.CityMedianImputer().fit(Z[tr], d.loc[tr, "City"])
centred = (Z[tr] - cm.transform(pd.DataFrame(np.nan, index=Z[tr].index, columns=Z.columns),
                                 d.loc[tr, "City"])).fillna(0)
s = np.linalg.svd(centred.values, compute_uv=False)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, len(s) + 1), s ** 2 / np.sum(s ** 2), marker="o")
ax.set_xlabel("component"); ax.set_ylabel("share of variance"); ax.set_title("Singular value spectrum (train)")
save("svd_spectrum")
print("variance share of first 3 components:", (s[:3] ** 2 / np.sum(s ** 2)).round(3))

variance share of first 3 components: [0.302 0.15  0.137]


**Reading the result.** The best method (city-centred KNN) reduces error to about 0.73-0.75 standard
deviations, versus 0.82-0.84 for the per-city median: a real but modest gain. Pollutant values on one day are only
partly predictable from the others. Plain KNN and SVD (no city information) are worse than the
median, because the biggest source of variation is *which city* a row is from. Higher SVD ranks
overfit: with mostly-missing rows, extra components fit noise.

Within city-centred KNN, error falls as k grows (20 -> 40 -> 80) but the gain is shrinking, so
k = 80 is used; larger k approaches a smooth average.

**Decision:** `city_knn` with k = 80 (`aq.CHOSEN_IMPUTER`). Limitation for the report: imputed
values are smooth estimates, and pollutants that a city never measured are filled entirely from
other cities' patterns. The `miss_<pollutant>` indicator columns record which values were imputed.

## 4.4 Features and targets

In [7]:
data, info = aq.build_dataset(df)
print("rows:", len(data), " RQ1 features:", len(info["rq1_features"]), " RQ2 features:", len(info["rq2_features"]))
print("missing values left in RQ2 features:", int(data[info["rq2_features"]].isna().sum().sum()))
print(info["rq2_features"])

rows: 25277  RQ1 features: 11  RQ2 features: 39
missing values left in RQ2 features: 0
['Z_PM2.5', 'Z_PM10', 'Z_NO', 'Z_NO2', 'Z_NOx', 'Z_NH3', 'Z_CO', 'Z_SO2', 'Z_O3', 'Z_Benzene', 'Z_Toluene', 'aqi_z_filled', 'aqi_missing', 'lag1_Z_PM2.5', 'lag1_Z_PM10', 'lag1_Z_NO2', 'lag1_aqi_z_filled', 'roll7_Z_PM2.5', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'city_Ahmedabad', 'city_Amaravati', 'city_Amritsar', 'city_Bengaluru', 'city_Brajrajnagar', 'city_Chennai', 'city_Delhi', 'city_Gurugram', 'city_Hyderabad', 'city_Jaipur', 'city_Jorapokhar', 'city_Kolkata', 'city_Lucknow', 'city_Patna', 'city_Talcher', 'city_Thiruvananthapuram', 'city_Visakhapatnam']


| Feature | Definition | Why |
|---|---|---|
| `Z_<pollutant>` | today's pollutant, log1p, standardised, imputed | RQ1 inputs, RQ2 inputs |
| `aqi_z_filled`, `aqi_missing` | today's log AQI (standardised); missing filled with city train median, flag | RQ2: today's level |
| `lag1_*` | yesterday's PM2.5, PM10, NO2, AQI, built **within each city** | autocorrelation (Step 3) |
| `roll7_Z_PM2.5` | mean of the last 7 days including today, within city | smooths single-day noise |
| `month_sin`, `month_cos` | cyclic month | December is next to January |
| `dow_sin`, `dow_cos` | cyclic day of week | small effect in Step 3; kept, cheap |
| `city_*` | one-hot city | city level differs strongly |

Targets: RQ1 `y_log_aqi` = log1p(today's AQI); RQ2 `y_unhealthy_next` = 1 if tomorrow's AQI > 200.
Rows with a missing target are dropped only inside the step that uses that target.

In [8]:
pos = (data.dropna(subset=["y_unhealthy_next"])
           .groupby("period")["y_unhealthy_next"].agg(rows="count", positive_rate="mean")
           .loc[["train", "val", "test"]])
nolock = data[(data["period"] == "test") & ~data["lockdown"]].dropna(subset=["y_unhealthy_next"])
pos.loc["test (no lockdown)"] = [len(nolock), nolock["y_unhealthy_next"].mean()]
pos = pos.round(3); pos.to_csv(TAB / "04_positive_rate.csv")
pos

,rows,positive_rate
period,,
train,13141.0,0.336
val,5921.0,0.245
test,2963.0,0.122
test (no lockdown),1473.0,0.210


## 4.5 Leakage checklist

* Log decision, means, standard deviations: train rows only.
* Imputer: fitted on train rows only, then applied to validation and test.
* AQI fill values (city medians): train rows only.
* Lags and rolling means: computed within each city, using only the current and earlier days.
  Step 2 confirmed there are no gaps in dates, so a one-row shift is a one-day shift.
* The RQ2 target is tomorrow's AQI; no feature uses tomorrow's data.
* Validation is used for all tuning; test is evaluated once per final model.

Known, accepted subtlety: the imputer fills a missing value using other pollutants *of the same
day*. For RQ2 that is fine (all are known today). For RQ1 it means an imputed PM2.5 can borrow
information from today's PM10, which is also a feature, never from AQI (AQI is not an imputer
input).